# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and explores complex human protein data from mass spectrometry analysis.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define Croissant dataset URL (schema)
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This provides a structured map of the data components. All entities are referenced by their `@id` field for consistency.

Let's list all record sets and show their available fields.

In [ ]:
record_sets = dataset.record_sets

print(f"Found {len(record_sets)} record sets in the dataset.\n")
for rs in record_sets:
    print(f"Record set name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) : {field.description}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. The record set and field `@id`s are referenced for precise data extraction. 

For demonstration, we will extract all record sets. For most analyses, you may want to select a particular record set based on the data overview above.

In [ ]:
# Extract data from all record sets
from collections import OrderedDict
dfs = OrderedDict()
for rs in record_sets:
    # Use @id to reference each record set
    data = list(dataset.records(record_set=rs.id))
    if data:
        df = pd.DataFrame(data)
    else:
        df = pd.DataFrame()
    dfs[rs.id] = df

# Display columns of the first record set with data
selected_rs_id = None
for rs_id, df in dfs.items():
    if not df.empty:
        selected_rs_id = rs_id
        print(f"Record set @id: {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
        break

if selected_rs_id is None:
    print('No non-empty record sets available for demonstration. Please check the dataset source.')

## 4. Exploratory Data Analysis (EDA)
We will demonstrate EDA on the first record set with non-empty data loaded above. This includes filtering by a numeric field, normalizing, and grouping (where possible). All fields are referenced by their `@id` as per Croissant.

> **Note:** Replace the variable values below (`numeric_field_id`, `group_field_id`) with the exact IDs from your data overview above if needed.

In [ ]:
# Use `selected_rs_id` and first numeric and group-able fields found
import numpy as np

df = dfs[selected_rs_id]

# Identify numeric fields based on dtype (float or int)
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) and not df[col].isna().all()]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    numeric_field_id = None
    print('No numeric fields found.')

if numeric_field_id is not None:
    # Set an arbitrary threshold at the mean
    threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize column
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to find a group-able (categorical/string) field
    group_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id]
    if group_candidates:
        group_field_id = group_candidates[0]
        print(f"Grouping by field: {group_field_id}")
        grouped = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean().sort_values(by=numeric_field_id, ascending=False)
        display(grouped.head())
    else:
        group_field_id = None
        print("No suitable group field found.")
else:
    print('No EDA performed as no numeric fields were found.')

## 5. Visualization
Visualize data distributions or relationships between fields. We demonstrate a histogram of the numeric variable and, if grouped, a bar plot by the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Average {numeric_field_id} by {group_field_id} (filtered)")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated loading the FAIR^2 mass spectrometry dataset using Croissant metadata, inspecting its structure, extracting data by `@id`, and performing basic EDA and visualization.

- The dataset contains multiple record sets, each with well-defined fields for protein and sample analysis.
- With precise references to record sets and fields by their `@id`, robust and unambiguous data workflows can be performed.
- Further analysis might involve deeper protein quantitation, outlier analysis, or integrating with external bioinformatics databases.